# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


## Setup — Run this cell first


In [1]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())


Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [2]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [3]:
# Task 1: Regional comparison bar chart
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))   # ascending, so the happiest region is at the top

print(region_avg)

# Highlight the top region, grey out the rest
top_region = region_avg.iloc[-1]['Region']
colors = ['#2a9d8f' if r == top_region else '#c0c7cf' for r in region_avg['Region']]

fig1 = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=colors,
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside'
))

fig1.update_layout(
    title=dict(
        text=f"<b>{top_region} is the happiest region, while Sub-Saharan Africa and South Asia trail far behind</b>",
        font=dict(size=18)
    ),
    xaxis=dict(title='Average happiness score (0–10)',
               range=[0, region_avg['Happiness_Score'].max() * 1.15],  # zero baseline
               showgrid=True, gridcolor='#eeeeee'),
    yaxis=dict(title=''),
    plot_bgcolor='white',
    font=dict(family='Arial', size=13),
    height=500,
    margin=dict(l=20, r=40, t=90, b=50)
)

fig1.show()

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [4]:
# Task 2: Top 8 vs. Bottom 8 contrast
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
gap = top8['Happiness_Score'].mean() - bottom8['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")
print(f"Gap between top 8 and bottom 8 averages: {gap:.2f}")

color_map = {'Top 8': '#2a9d8f', 'Bottom 8': '#e76f51'}
bar_colors = combined['Group'].map(color_map)

fig2 = go.Figure(go.Bar(
    x=combined['Happiness_Score'],
    y=combined['Country'],
    orientation='h',
    marker_color=bar_colors,
    text=combined['Happiness_Score'].round(2),
    textposition='outside',
    showlegend=False
))

# Legend entries for the two groups
for group, col in color_map.items():
    fig2.add_trace(go.Bar(x=[None], y=[None], orientation='h',
                          marker_color=col, name=group))

# Global average reference line (stretch goal)
fig2.add_vline(x=global_avg, line_dash='dash', line_color='#444444',
               annotation_text=f"Global avg: {global_avg:.2f}",
               annotation_position='top')

# Separator between the two groups (bars 0-7 are Bottom 8, 8-15 are Top 8)
fig2.add_hline(y=7.5, line_dash='dot', line_color='#888888')

# Gap annotation
fig2.add_annotation(
    x=combined['Happiness_Score'].min() + 0.3, y=11.5,
    text=f"<b>Gap: {gap:.1f} points</b><br>between the group averages",
    showarrow=False, font=dict(size=14, color='#333333'),
    bgcolor='#f6f6f6', bordercolor='#cccccc', borderpad=6
)

fig2.update_layout(
    title=dict(
        text=f"<b>The happiest countries score about {gap:.1f} points higher than the least happy</b>",
        font=dict(size=18)
    ),
    xaxis=dict(title='Happiness score (0–10)',
               range=[0, combined['Happiness_Score'].max() * 1.12],  # zero baseline
               showgrid=True, gridcolor='#eeeeee'),
    yaxis=dict(title='', categoryorder='array',
               categoryarray=combined['Country'].tolist()),
    plot_bgcolor='white',
    font=dict(family='Arial', size=13),
    legend=dict(title='', orientation='h', y=-0.12, x=0.5, xanchor='center'),
    height=700,
    margin=dict(l=20, r=40, t=90, b=70)
)

fig2.show()

Global average: 5.81
Gap between top 8 and bottom 8 averages: 4.16


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [5]:
# Stretch goal: grouped bar chart (GDP vs Freedom)
regions = ['Western Europe', 'Latin America', 'East Asia',
           'Sub-Saharan Africa', 'South Asia']

# Safety check: warn if a region name isn't in your data
missing = [r for r in regions if r not in df['Region'].unique()]
if missing:
    print("Not found in dataset (check spelling):", missing)

summary = (df[df['Region'].isin(regions)]
           .groupby('Region')[['GDP', 'Freedom']]
           .mean()
           .reindex([r for r in regions if r not in missing]))

fig3 = go.Figure()
fig3.add_trace(go.Bar(name='GDP', x=summary.index, y=summary['GDP'],
                      marker_color='#264653',
                      text=summary['GDP'].round(2), textposition='outside'))
fig3.add_trace(go.Bar(name='Freedom', x=summary.index, y=summary['Freedom'],
                      marker_color='#e9c46a',
                      text=summary['Freedom'].round(2), textposition='outside'))

fig3.update_layout(
    barmode='group',
    title=dict(
        text="<b>Economic strength and freedom rise together: Western Europe leads, South Asia and Sub-Saharan Africa lag</b>",
        font=dict(size=17)
    ),
    xaxis=dict(title=''),
    yaxis=dict(title='Average contribution to happiness score',
               range=[0, summary.max().max() * 1.2],  # zero baseline
               gridcolor='#eeeeee'),
    plot_bgcolor='white',
    font=dict(family='Arial', size=13),
    legend=dict(title='Sub-factor'),
    height=550,
    margin=dict(l=20, r=40, t=100, b=50)
)

fig3.show()

Not found in dataset (check spelling): ['Latin America']
